In [ ]:
# from huggingface_hub import snapshot_download

# snapshot_download(
#     repo_id=model_name,         # 你要下载的模型
#     local_dir=cache_dir,  # 保存路径
#     local_dir_use_symlinks=False         # 关闭软链接，直接复制真实文件
# )
# !pip install accelerate


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# model_name = "Qwen/Qwen2.5-0.5B"  # 替换为你要下载的模型
cache_dir = "/home/lishengping/qwen2.5_0.5B"   # 你希望保存模型的本地路径
model = AutoModelForCausalLM.from_pretrained(
    cache_dir,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(cache_dir)

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [2]:
# prompt = "Give me a short introduction to large language model."
# messages = [
#     {"role": "user", "content": prompt}
# ]
# text = tokenizer.apply_chat_template(
#     messages,
#     tokenize=False,
#     add_generation_prompt=True
# )
# model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
# generated_ids = model.generate(
#     **model_inputs,
#     max_new_tokens=512
# )
# generated_ids = [
#     output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
# ]
# response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


In [ ]:
# # torch
# model.norm.weight torch.Size([896])
# model.embed_tokens.weight torch.Size([151936, 896])
# model.layers.{layer_index}.mlp.up_proj.weight torch.Size([4864, 896])
# model.layers.{layer_index}.mlp.gate_proj.weight torch.Size([4864, 896])
# model.layers.{layer_index}.mlp.down_proj.weight torch.Size([896, 4864])
# model.layers.{layer_index}.input_layernorm.weight torch.Size([896])
# model.layers.{layer_index}.post_attention_layernorm.weight torch.Size([896])
# model.layers.{layer_index}.self_attn.q_proj.weight torch.Size([896, 896])
# model.layers.{layer_index}.self_attn.q_proj.bias torch.Size([896])
# model.layers.{layer_index}.self_attn.k_proj.weight torch.Size([128, 896])
# model.layers.{layer_index}.self_attn.k_proj.bias torch.Size([128])
# model.layers.{layer_index}.self_attn.v_proj.weight torch.Size([128, 896])
# model.layers.{layer_index}.self_attn.v_proj.bias torch.Size([128])
# model.layers.{layer_index}.self_attn.o_proj.weight torch.Size([896, 896])
# # jax
# params.decoder.decoder_norm.scale: (896,)
# params.token_embedder.embedding: (151936, 896)
# params.decoder.{layer_index}.sub_0.mlp.wi_0.kernel: (896, 4864)
# params.decoder.{layer_index}.sub_0.mlp.wi_1.kernel: (896, 4864)
# params.decoder.{layer_index}.sub_0.mlp.wo.kernel: (4864, 896)
# params.decoder.{layer_index}.sub_0.pre_self_attention_layer_norm.scale: (896,)
# params.decoder.{layer_index}.sub_0.post_self_attention_layer_norm.scale: (896,)
# params.decoder.{layer_index}.sub_0.self_attention.query.kernel: (896, 14, 64)
# params.decoder.{layer_index}.sub_0.self_attention.query.bias: (14, 64)
# params.decoder.{layer_index}.sub_0.self_attention.key.kernel: (896, 2, 64)
# params.decoder.{layer_index}.sub_0.self_attention.key.bias: (2, 64)
# params.decoder.{layer_index}.sub_0.self_attention.value.bias: (2, 64)
# params.decoder.{layer_index}.sub_0.self_attention.value.kernel: (896, 2, 64)
# params.decoder.{layer_index}.sub_0.self_attention.out.kernel: (14, 64, 896)

In [ ]:
# !pip install tensorflow==2.16.1
import json
import os
import sys
import asyncio
import argparse
from collections import defaultdict
import time

os.environ["JAX_PLATFORMS"] = "cpu"

import torch
import numpy as np
import jax
import orbax
import orbax.checkpoint as ocp
from etils import epath
from jax.sharding import PartitionSpec as PS
from flax.traverse_util import flatten_dict, unflatten_dict
import base64


def decode_base64(encoded_str):
    decoded_bytes = base64.b64decode(encoded_str)
    decoded_str = decoded_bytes.decode('utf-8')
    return decoded_str

def encode_base64(decoded_str):
    # decoded_str = "opt_state.mu.params.token_embedder.embedding"
    encoded_string = base64.b64encode(decoded_str.encode('utf-8')).decode('utf-8')
    return encoded_string

# 新保存一个jax版本的模型，从中读取模型的keys，之后也会基于这个模型对参数进行转换
_sharding_path = 'gs://newproject-1-llm_base_models_europe-west4/qwen2.5/_sharding'
_sharding_path = epath.Path(_sharding_path)
with _sharding_path.open('r') as f:
    _sharding = json.load(f)

model_shardings = {}
for k, v in _sharding.items():
    base_k = decode_base64(k)
    base_k_split = tuple(base_k.split('.'))
    if 'opt_state' in base_k or 'step' in base_k: continue
    print(base_k_split)
    model_shardings[base_k_split] = v
    

In [ ]:
# load model
# 基于读取的jax keys，构造shape dtype 文件，便于load jax model
# 0.5B
vocab_size = 151936
base_emb_dim = 896
head_dim = 64
base_num_query_heads = 14
base_num_kv_heads = 2
base_mlp_dim = 4864
base_num_decoder_layers = 24


# 3B
vocab_size = 151936
base_emb_dim = 2048
head_dim = 128
base_num_query_heads = 16
base_num_kv_heads = 2
base_mlp_dim = 11008
base_num_decoder_layers = 36

mesh_axes = ['data', 'stage', 'fsdp', 'fsdp_transpose', 'sequence', 'tensor', 'tensor_transpose', 'tensor_sequence', 'expert', 'autoregressive']
devices = np.asarray(jax.devices()).reshape([1] * len(mesh_axes))
mesh = jax.sharding.Mesh(devices, mesh_axes)
sharding = jax.sharding.NamedSharding(mesh, PS()) # Sharding is None because we use cpu to load weights
weight_dtype = np.float32 # set restore weights dtype, np.float32 or np.float16
abstract_unboxed_params = {}
for k, v in model_shardings.items():
    jointk = '.'.join(k)
    if 'embedding' in jointk:
        shape = (vocab_size, base_emb_dim)
    elif 'scale' in jointk:
        shape = (base_emb_dim, )
    elif 'mlp.wi_' in jointk:
        shape = (base_emb_dim, base_mlp_dim)
    elif 'mlp.wo.' in jointk:
        shape = (base_mlp_dim, base_emb_dim)
    elif 'query.kernel' in jointk:
        shape = (base_emb_dim, base_num_query_heads, head_dim)
    elif 'query.bias' in jointk:
        shape = (base_num_query_heads, head_dim, )
    elif 'key.kernel' in jointk or 'value.kernel' in jointk:
        shape = (base_emb_dim, base_num_kv_heads, head_dim)
    elif 'key.bias' in jointk  or 'value.bias' in jointk:
        shape = (base_num_kv_heads, head_dim, )
    elif 'out.kernel' in jointk:
        shape = (base_num_query_heads, head_dim, base_emb_dim)
    else:
        print(f'Unmatched params name: {jointk}')
    print(k, shape)
    abstract_unboxed_params[k] = jax.ShapeDtypeStruct(shape=shape, dtype=weight_dtype, sharding=sharding)           
abstract_unboxed_params = unflatten_dict(abstract_unboxed_params)

jax_model_path = 'gs://newproject-1-llm_base_models_europe-west4/v5p_256/7B/test/checkpoints/250/items'
ckpt = epath.Path(jax_model_path)
ckptr = ocp.PyTreeCheckpointer()
restore_args = ocp.checkpoint_utils.construct_restore_args(abstract_unboxed_params)

restored = ckptr.restore(
  ckpt, item=abstract_unboxed_params, transforms={}, restore_args=restore_args
)

('params', 'params', 'token_embedder', 'embedding') (151936, 896)
('params', 'params', 'decoder', 'decoder_norm', 'scale') (896,)
('params', 'params', 'decoder', 'layers_4', 'sub_0', 'mlp', 'wi_0', 'kernel') (896, 4864)
('params', 'params', 'decoder', 'layers_4', 'sub_0', 'mlp', 'wi_1', 'kernel') (896, 4864)
('params', 'params', 'decoder', 'layers_4', 'sub_0', 'mlp', 'wo', 'kernel') (4864, 896)
('params', 'params', 'decoder', 'layers_4', 'sub_0', 'post_self_attention_layer_norm', 'scale') (896,)
('params', 'params', 'decoder', 'layers_4', 'sub_0', 'pre_self_attention_layer_norm', 'scale') (896,)
('params', 'params', 'decoder', 'layers_4', 'sub_0', 'self_attention', 'out', 'kernel') (14, 64, 896)
('params', 'params', 'decoder', 'layers_4', 'sub_0', 'self_attention', 'key', 'bias') (2, 64)
('params', 'params', 'decoder', 'layers_4', 'sub_0', 'self_attention', 'key', 'kernel') (896, 2, 64)
('params', 'params', 'decoder', 'layers_4', 'sub_0', 'self_attention', 'query', 'bias') (14, 64)
('p

In [190]:
qwen_jax2torch = {
    'params.token_embedder.embedding': 'model.embed_tokens.weight',  
    'params.decoder.decoder_norm.scale': 'model.norm.weight',
    'params.decoder.layers_0.sub_0.mlp.wi_1.kernel': 'model.layers.0.mlp.up_proj.weight', # .T
    'params.decoder.layers_0.sub_0.mlp.wi_0.kernel': 'model.layers.0.mlp.gate_proj.weight', # .T
    'params.decoder.layers_0.sub_0.mlp.wo.kernel': 'model.layers.0.mlp.down_proj.weight', # .T
    'params.decoder.layers_0.sub_0.post_self_attention_layer_norm.scale': 'model.layers.0.post_attention_layernorm.weight',
    'params.decoder.layers_0.sub_0.pre_self_attention_layer_norm.scale': 'model.layers.0.input_layernorm.weight',
    'params.decoder.layers_0.sub_0.self_attention.query.kernel': 'model.layers.0.self_attn.q_proj.weight', # .T.reshape(base_emb_dim, base_num_query_heads, head_dim)
    'params.decoder.layers_0.sub_0.self_attention.query.bias': 'model.layers.0.self_attn.q_proj.bias', # .reshape(base_num_query_heads, head_dim)
    'params.decoder.layers_0.sub_0.self_attention.key.kernel': 'model.layers.0.self_attn.k_proj.weight', # .T.reshape(base_emb_dim, base_num_kv_heads, head_dim)
    'params.decoder.layers_0.sub_0.self_attention.key.bias': 'model.layers.0.self_attn.k_proj.bias', # .reshape(base_num_kv_heads, head_dim)
    'params.decoder.layers_0.sub_0.self_attention.value.kernel': 'model.layers.0.self_attn.v_proj.weight', # .T.reshape(base_emb_dim, base_num_kv_heads, head_dim)
    'params.decoder.layers_0.sub_0.self_attention.value.bias': 'model.layers.0.self_attn.v_proj.bias', # # .reshape(base_num_kv_heads, head_dim)
    'params.decoder.layers_0.sub_0.self_attention.out.kernel': 'model.layers.0.self_attn.o_proj.weight', # .T.reshape(base_num_query_heads, head_dim, base_emb_dim)
    
}

all_layers_qwen_jax2torch = {}

for k, v in qwen_jax2torch.items():
    if 'layers_0' in k:
        for l in range(base_num_decoder_layers):
            newk = k.replace('layers_0', f'layers_{l}')
            newv = v.replace('layers.0', f'layers.{l}')
            all_layers_qwen_jax2torch[newk] = newv
    else:
        all_layers_qwen_jax2torch[k] = v

convert_params = {}
for k, v in flatten_dict(restored['params']).items():
    jax_key = '.'.join(k)
    torch_key = all_layers_qwen_jax2torch[jax_key]
    torch_value = model.state_dict()[torch_key]
    if '.mlp.' in jax_key:
        torch_value = torch_value.T
    elif 'query.kernel' in jax_key:
        torch_value = torch_value.T.reshape(base_emb_dim, base_num_query_heads, head_dim)
    elif 'query.bias' in jax_key:
        torch_value = torch_value.reshape(base_num_query_heads, head_dim)   
    elif 'key.kernel' in jax_key or 'value.kernel' in jax_key:
        torch_value = torch_value.T.reshape(base_emb_dim, base_num_kv_heads, head_dim)
    elif 'key.bias' in jax_key or 'value.bias' in jax_key:
        torch_value = torch_value.reshape(base_num_kv_heads, head_dim)    
    elif 'out.kernel' in jax_key:
        torch_value = torch_value.T.reshape(base_num_query_heads, head_dim, base_emb_dim)
    assert v.shape == torch_value.shape
    convert_params[k] = jnp.array(torch_value.to(torch.float32), dtype=jnp.bfloat16)
flatten_convert_params = unflatten_dict(convert_params)

In [191]:
# save model
checkpoint_dir = 'gs://newproject-1-llm_base_models_europe-west4/v5p_256/7B/qwen2.5_3B_torch2jax_0425/checkpoints/0/items'
orbax_checkpointer = ocp.PyTreeCheckpointer()
orbax_checkpointer.save(checkpoint_dir, {"params": flatten_convert_params}, force=True)
print(f"Quantized params checkpoint saved at: {checkpoint_dir}")

Quantized params checkpoint saved at: gs://newproject-1-llm_base_models_europe-west4/v5p_256/7B/qwen2.5_3B_torch2jax_0425/checkpoints/0/items


In [189]:
# tokenizer2 = AutoTokenizer.from_pretrained('Qwen/Qwen-14B')